In [ ]:
# ==========================================
# 1. IMPORT REQUIRED LIBRARIES
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# ==========================================
# 2. LOAD DATASET
# ==========================================

df = pd.read_csv("emails.csv")

print("First 5 rows:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nMissing Values:")
print(df.isnull().sum().sum())

# ==========================================
# 3. DATA PREPROCESSING
# ==========================================

# Remove unnecessary columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# Remove email ID column
df = df.drop(columns=["Email"], errors="ignore")

# Target column is Prediction
# If not available, use the last column
target = "Prediction" if "Prediction" in df.columns else df.columns[-1]

# Convert all columns to numeric values
df = df.apply(pd.to_numeric, errors="coerce")

# Replace missing values with 0
df = df.fillna(0)

# Separate input and output
X = df.drop(target, axis=1)
y = df[target].astype(int)

print("\nInput Features:", X.shape)
print("Target Values:")
print(y.value_counts())

# ==========================================
# 4. SPLIT DATASET
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Data:", X_train.shape)
print("Testing Data:", X_test.shape)

# ==========================================
# 5. K-NEAREST NEIGHBORS (KNN)
# ==========================================

# Standardize data and apply KNN
knn = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5)
)

# Train model
knn.fit(X_train, y_train)

# Predict results
knn_pred = knn.predict(X_test)

# Evaluate KNN
print("\n===== KNN CLASSIFICATION =====")

print("Accuracy:", accuracy_score(y_test, knn_pred))
print("Precision:", precision_score(y_test, knn_pred))
print("Recall:", recall_score(y_test, knn_pred))
print("F1 Score:", f1_score(y_test, knn_pred))

print("\nClassification Report:")
print(classification_report(
    y_test, knn_pred,
    target_names=["Not Spam", "Spam"]
))

# ==========================================
# 6. SUPPORT VECTOR MACHINE (SVM)
# ==========================================

# Standardize data and apply SVM
svm = make_pipeline(
    StandardScaler(),
    SVC(kernel="linear")
)

# Train model
svm.fit(X_train, y_train)

# Predict results
svm_pred = svm.predict(X_test)

# Evaluate SVM
print("\n===== SVM CLASSIFICATION =====")

print("Accuracy:", accuracy_score(y_test, svm_pred))
print("Precision:", precision_score(y_test, svm_pred))
print("Recall:", recall_score(y_test, svm_pred))
print("F1 Score:", f1_score(y_test, svm_pred))

print("\nClassification Report:")
print(classification_report(
    y_test, svm_pred,
    target_names=["Not Spam", "Spam"]
))

# ==========================================
# 7. CONFUSION MATRIX FOR KNN
# ==========================================

cm_knn = confusion_matrix(y_test, knn_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm_knn, annot=True, fmt="d", cmap="Blues")

plt.title("KNN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# ==========================================
# 8. CONFUSION MATRIX FOR SVM
# ==========================================

cm_svm = confusion_matrix(y_test, svm_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm_svm, annot=True, fmt="d", cmap="Greens")

plt.title("SVM Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# ==========================================
# 9. COMPARE BOTH MODELS
# ==========================================

results = pd.DataFrame({
    "Model": ["KNN", "SVM"],

    "Accuracy": [
        accuracy_score(y_test, knn_pred),
        accuracy_score(y_test, svm_pred)
    ],

    "Precision": [
        precision_score(y_test, knn_pred),
        precision_score(y_test, svm_pred)
    ],

    "Recall": [
        recall_score(y_test, knn_pred),
        recall_score(y_test, svm_pred)
    ],

    "F1 Score": [
        f1_score(y_test, knn_pred),
        f1_score(y_test, svm_pred)
    ]
})

print("\n===== MODEL COMPARISON =====")
print(results)

# ==========================================
# 10. PERFORMANCE COMPARISON GRAPH
# ==========================================

results.set_index("Model").plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title("KNN vs SVM Performance")
plt.ylabel("Score")
plt.ylim(0, 1.1)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.show()

# ==========================================
# 11. PREDICT A SAMPLE EMAIL
# ==========================================

sample = X_test.iloc[[0]]

knn_result = knn.predict(sample)[0]
svm_result = svm.predict(sample)[0]

print("\n===== EMAIL PREDICTION =====")

print("KNN Prediction:",
      "Spam" if knn_result == 1 else "Not Spam")

print("SVM Prediction:",
      "Spam" if svm_result == 1 else "Not Spam")